# INSTALLATION, ETC.

In [1]:
!pip install langchain langchain-anthropic anthropic 

# IMPORTS

In [2]:
%load_ext autoreload
%autoreload 2

import warnings
def warn(*args, **kwargs):
    pass
warnings.warn = warn
warnings.filterwarnings('ignore')

import os, subprocess
from IPython.display import Markdown, display

## SET UP CLAUDE KEY

In [3]:
## extract anthropic api key from zsh environment and set it in os.environ for langchain-anthropic to work
key = subprocess.check_output(
    "zsh -lc 'source ~/.zshrc >/dev/null 2>&1; printf %s \"$ANTHROPIC_API_KEY\"'",
    shell=True,
    text=True
).strip()

os.environ["ANTHROPIC_API_KEY"] = key
print(bool(key))

True


### SETUP CONNECTION AND TEST CONNECTION

In [4]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
    model="claude-sonnet-4-6",
    temperature=0.2
)

resp = llm.invoke("Say hello in one short sentence.")
display(Markdown(f"**Response from Claude Sonnet 4-6:**\n\n{resp.content}"))

**Response from Claude Sonnet 4-6:**

Hello! Hope you're having a wonderful day! 😊

### SETUP MEMORY AND TEST MEMORY CREATION

In [5]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

chat_history = InMemoryChatMessageHistory()
conversation = RunnableWithMessageHistory(llm, lambda session_id: chat_history)
config = {"configurable": {"session_id": "demo"}}


In [6]:
# Test memory # 1
display(Markdown("**Testing memory with a conversation:**"))
content = conversation.invoke('I\'m studying transformers for NLP, exam Friday', config=config).content
display(Markdown(f"**Claude Sonnet 4-6 response:**\n\n{content}"))

**Testing memory with a conversation:**

**Claude Sonnet 4-6 response:**

# Transformer Study Guide for NLP

## Core Architecture

### The Big Picture
**Input → Encoder → Decoder → Output**
- Encoder: processes input sequence
- Decoder: generates output sequence
- Both use **stacked layers** (typically 6)

---

## Key Components

### 1. Self-Attention
The core mechanism - lets each word "look at" other words

**Formula:**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- **Q** (Query) - what am I looking for?
- **K** (Key) - what do I contain?
- **V** (Value) - what do I actually pass forward?
- **√d_k** scaling prevents vanishing gradients in softmax

### 2. Multi-Head Attention
- Run attention **h times in parallel** with different learned projections
- Captures different types of relationships simultaneously
- Concatenate → linear projection

### 3. Positional Encoding
- Transformers have **no recurrence** → need to inject position info
- Added to embeddings (sin/cos functions or learned)

### 4. Feed-Forward Network
- Applied **position-wise** (same weights, each position independently)
- Two linear layers + ReLU

---

## Encoder vs Decoder

| Feature | Encoder | Decoder |
|---------|---------|---------|
| Self-attention | Bidirectional | **Masked** (causal) |
| Cross-attention | ❌ | ✅ (attends to encoder) |
| Use case | BERT | GPT, translation |

---

## Common Exam Topics

### Why Transformers > RNNs?
- **Parallelizable** (no sequential dependency)
- Better **long-range dependencies**
- More scalable

### Masked Self-Attention (Decoder)
- Prevents looking at **future tokens** during training
- Achieved by setting future positions to **-∞** before softmax

### Layer Normalization
- Applied **before** or after each sub-layer
- Stabilizes training

---

## Famous Models to Know

| Model | Type | Key Feature |
|-------|------|-------------|
| **BERT** | Encoder-only | Bidirectional, MLM pre-training |
| **GPT** | Decoder-only | Autoregressive, causal |
| **T5/BART** | Encoder-Decoder | Seq2seq tasks |

---

## Quick Formulas

- Attention complexity: **O(n²·d)** — quadratic in sequence length
- Parameters in attention head: **3·d²** (for Q, K, V projections)

---

## What to Focus On

1. ✅ Draw the architecture from memory
2. ✅ Explain attention in plain English
3. ✅ Know why masking is needed
4. ✅ BERT vs GPT differences
5. ✅ Positional encoding purpose

---

**What specific part do you want to go deeper on?** Happy to do practice questions too!

In [7]:
# Test memory # 2
content1 = conversation.invoke("What am I studying?", config=config).content
display(Markdown(f"**Claude Sonnet 4-6 response:**\n\n{content1}"))

**Claude Sonnet 4-6 response:**

You're studying **Transformers for NLP** (Natural Language Processing), specifically for an exam on **Friday**.

Want me to quiz you on anything to help you prepare?

# SETUP TOOL AND MEMORY TOGETHER

In [8]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

## Simple decorated calculator function to evaluate simple mathematical expressions
@tool
def calculator(expression: str) -> str:
    """A calculator that can evaluate simple mathematical expressions."""
    try:
        result = str(eval(expression, {"__builtin__": {}}, {}))
        return result
    except Exception as e:
        return f"Error: {str(e)}"
    
print("\n--- Calculator Test ---\n")
print(calculator("2 + 3 * (4 - 1)"))
print(calculator("10 / 0"))
print(calculator("invalid expression")) 


--- Calculator Test ---

11
Error: division by zero
Error: invalid syntax (<string>, line 1)


In [9]:
## Bind tool to the llm to create a new llm that can use the tool when needed
llm_with_tools = llm.bind_tools([calculator])

# separate history for the tool-enabled conversation so that we can see the difference in responses when the tool is used
tool_history = InMemoryChatMessageHistory()  

# Mapping of tool names to tool functions for the tool-enabled conversation
tools_by_name = {'calculator': calculator}

In [10]:
def run_react_loop( user_input ):
    # Add user message to history
    tool_history.add_message(HumanMessage(content=user_input))

    while (True):
        # Call LLM with full conversation history
        response = llm_with_tools.invoke(tool_history.messages)
        
        # history may contain tool use or no tool use. 
        # If tool use, 
        #   the tool response will be added to the history and then the 
        #   loop will continue and call the LLM again with the updated history. 
        # If no tool use, 
        #   then we can break the loop and display the final response.
        tool_history.add_message(response)

        # If no tools were called 
        if not response.tool_calls:
            return response.content
        
        # Execute each tool call and add the tool response to the history
        for tool_call in response.tool_calls:
            result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
            tool_history.add_message(
                ToolMessage(
                    content=str(result),
                    tool_call_id = tool_call["id"] 
                    )
                )
        # LLM can now see the tool response in the updated history and can decide to call more tools or not call any tools and respond to the user.


In [11]:
response = run_react_loop("What is 12 multiplied by 8?")
display(Markdown(f"**Final Response after tool use:** {response}"))

**Final Response after tool use:** 12 multiplied by 8 equals **96**.

In [12]:
response = run_react_loop("I'm studying transformers. How many hours until Friday exam if today is Wednesday and I study 2 hours per day?")
display(Markdown(f"**Response:**\n\n{response}"))

**Response:**

If you study **2 hours per day** over the next **2 days** (Wednesday and Thursday), you'll have **4 hours** of study time before your Friday exam.

Make the most of those 4 hours — good luck studying transformers! 🎓

# MULTITURN CONVERSATION

In [13]:
turns = ["Hi, I’m studying transformers for NLP, and my exam is on Friday.",
    "Can you suggest a 2-hour study plan for tonight?",
    "If today is Wednesday, how many hours until my exam on Friday if I study 2 hours per day?",
    "What topic am I studying again?",
    "When is my exam?"]

In [14]:
for msg in turns:
    response = conversation.invoke(msg, config=config).content
    display(Markdown(f"**User:** {msg}\n\n**Claude Sonnet 4-6:** {response}"))

print("\n--- End of Conversation ---\n")

**User:** Hi, I’m studying transformers for NLP, and my exam is on Friday.

**Claude Sonnet 4-6:** Hi! You mentioned that earlier 😊

You're studying **Transformers for NLP** with an exam on **Friday**.

Would you like to:
- **Review** specific concepts?
- **Practice questions** to test yourself?
- **Clarify** anything confusing?

What would be most helpful right now?

**User:** Can you suggest a 2-hour study plan for tonight?

**Claude Sonnet 4-6:** # 2-Hour Transformer Study Plan

## ⏱️ Hour 1 - Core Concepts (Understanding)

### 0:00 - 0:20 | Architecture Overview
- Draw the full transformer diagram from memory
- Encoder vs Decoder differences
- How data flows through the model

### 0:20 - 0:45 | Attention Mechanism
- Self-attention formula (Q, K, V)
- Why the √d_k scaling?
- Multi-head attention purpose

### 0:45 - 1:00 | Supporting Components
- Positional encoding (why it's needed)
- Layer normalization
- Feed-forward networks

---

## ⏱️ Hour 2 - Application & Testing (Recall)

### 1:00 - 1:20 | Key Models
- BERT vs GPT vs T5
- When to use each
- Pre-training objectives (MLM, causal LM)

### 1:20 - 1:40 | Practice Questions
- Answer questions out loud or written
- Focus on **weak areas** from Hour 1

### 1:40 - 2:00 | Review & Consolidate
- Recheck anything unclear
- Summarize key points in your own words

---

## 💡 Tips
- **No passive reading** - actively recall
- Take a **5 min break** between hours
- Flag weak spots as you go

---

Want me to **generate practice questions** to use in that 1:20 slot?

**User:** If today is Wednesday, how many hours until my exam on Friday if I study 2 hours per day?

**Claude Sonnet 4-6:** I can tell you it's **2 days until Friday**, but I don't know:

- **What time it is now** (Wednesday)
- **What time your exam is on Friday**

### What I can say:
- You have **2 study sessions** planned (Wed + Thu)
- That's **4 hours total** of study time before the exam

---

Could you tell me:
1. What time is it now?
2. What time is your exam Friday?

Then I can give you the exact hours! 🕐

**User:** What topic am I studying again?

**Claude Sonnet 4-6:** You're studying **Transformers for NLP** with an exam on **Friday**! 🎓

Need to get back to studying? 😄

**User:** When is my exam?

**Claude Sonnet 4-6:** Your exam is on **Friday**! 📅

Ready to get back to studying? 😊


--- End of Conversation ---



# STATELESS / MEMORY LESS TEST 

In [15]:
print("=== STATELESS (no memory) ===")
stateless_llm = ChatAnthropic(
    model="claude-sonnet-4-6",
    temperature=0.2
)

# Turn 1
resp1 = stateless_llm.invoke("I'm studying transformers for NLP, exam Friday")
display(Markdown(f"**Response to Turn 1 from Claude Sonnet 4-6 (Stateless):** {resp1.content}"))

# Turn 2 - model has no memory of Turn 1
resp2 = stateless_llm.invoke("What am I studying?")
display(Markdown(f"**Response to Turn 2 from Claude Sonnet 4-6 (Stateless):** {resp2.content}"))

print("\n=== END STATELESS ===")

=== STATELESS (no memory) ===


**Response to Turn 1 from Claude Sonnet 4-6 (Stateless):** # Transformer Study Guide for NLP

## Core Architecture

### The Big Picture
```
Input → Embedding + Positional Encoding → Encoder Stack → Decoder Stack → Output
```

---

## Key Components to Know

### 1. **Self-Attention (Most Important)**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- **Q** = Query, **K** = Key, **V** = Value
- **√d_k** scaling prevents vanishing gradients in softmax
- Allows each token to "look at" all other tokens

### 2. **Multi-Head Attention**
- Run attention **h times** in parallel with different learned projections
- Captures different types of relationships simultaneously
- Concatenate → linear projection

### 3. **Positional Encoding**
- Transformers have **no inherent word order**
- Add sinusoidal (or learned) encodings to embeddings
- Gives the model sequence position information

### 4. **Feed-Forward Network**
- Applied to each position **independently**
- Two linear layers + ReLU
- Same weights across positions

---

## Encoder vs Decoder

| Feature | Encoder | Decoder |
|---------|---------|---------|
| Attention type | Bidirectional | Masked (causal) |
| Sees | Full sequence | Past tokens only |
| Example model | BERT | GPT |
| Task | Understanding | Generation |

---

## Common Exam Topics

- Why **masking** in decoder? (prevent cheating on future tokens)
- Why **layer norm**? (training stability)
- **Residual connections** → help gradient flow
- Complexity: **O(n²)** in sequence length

---

## Quick Model Reference

| Model | Type | Use Case |
|-------|------|----------|
| BERT | Encoder only | Classification, QA |
| GPT | Decoder only | Text generation |
| T5/BART | Encoder-Decoder | Translation, summarization |

---

## What do you want to dive deeper into?
- Attention math?
- BERT vs GPT differences?
- Training objectives (MLM, CLM)?
- Specific architectures?

**Response to Turn 2 from Claude Sonnet 4-6 (Stateless):** I don't have any information about you or what you're studying. Could you tell me more? I'd be happy to help with whatever subject or topic you're working on! 😊


=== END STATELESS ===
